In [1]:
import time
start = time.time()

# Package install/import

In [2]:
#!pip install -U sentence-transformers
#!pip install umap-learn
#!pip install clean-text
#!pip install glob
#!pip install rake-nltk
#!pip install sumy

In [3]:
from sentence_transformers import SentenceTransformer
import operator
import regex as re
from pandas import DataFrame
import numpy as np
import umap
import hdbscan
import matplotlib.pyplot as plt
import pandas as pd
import PyPDF2
import cleantext
from cleantext import clean
import glob
import nltk
import rake_nltk
from rake_nltk import Rake
from transformers import pipeline
import os
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lex_rank import LexRankSummarizer

nltk.download('stopwords')

Since the GPL-licensed package `unidecode` is not installed, using Python's `unicodedata` package which yields worse results.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\rdominguez\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

# Data import

### Local Import

In [4]:
pdf_files = [file for file in glob.glob(r'C:\Users\rdominguez\Documents\Pers\UChicago\Capstone\Data\PDF\*.pdf')]
print(pdf_files)

big_chungus_vector = []
for pdfs in pdf_files:
    pdfFileObj = open(pdfs,'rb')
    pdfReader = PyPDF2.PdfFileReader(pdfFileObj)
    pagecount = pdfReader.getNumPages ()
    for count in range(pagecount):
        page = pdfReader.getPage(count)
        page_extract = page.extractText ()
        row = [pdfs, count, page_extract]
        big_chungus_vector.append(row)
        
#big_chungus_vector

['C:\\Users\\rdominguez\\Documents\\Pers\\UChicago\\Capstone\\Data\\PDF\\HMH_IntoReading_G5_TeachersGuide_v01.pdf', 'C:\\Users\\rdominguez\\Documents\\Pers\\UChicago\\Capstone\\Data\\PDF\\HMH_IntoReading_G5_TeachersGuide_v02.pdf', 'C:\\Users\\rdominguez\\Documents\\Pers\\UChicago\\Capstone\\Data\\PDF\\HMH_IntoReading_G5_TeachersGuide_v03.pdf', 'C:\\Users\\rdominguez\\Documents\\Pers\\UChicago\\Capstone\\Data\\PDF\\HMH_IntoReading_G5_TeachersGuide_v04.pdf', 'C:\\Users\\rdominguez\\Documents\\Pers\\UChicago\\Capstone\\Data\\PDF\\HMH_IntoReading_G5_TeachersGuide_v05.pdf', 'C:\\Users\\rdominguez\\Documents\\Pers\\UChicago\\Capstone\\Data\\PDF\\HMH_IntoReading_G5_TeachersGuide_v06.pdf']


### Colab import

# Data clean up

In [5]:
big_chungus_df = DataFrame(big_chungus_vector)
big_chungus_df.columns =['file', 'page', 'text']
big_chungus_df['page'] += 1 ## so the element count matches actual page numbers
#big_chungus_df.to_csv('grade5_corpus.csv',index=False)

print(len(big_chungus_df))

#remove null pages
big_chungus_df['text'] = big_chungus_df['text'].replace('', np.nan, inplace=False)
big_chungus_df = big_chungus_df[big_chungus_df['text'].notnull()]
big_chungus_df = big_chungus_df.reset_index()
print(len(big_chungus_df))

#clean
big_chungus_df['text_clean'] = [item.lower() for item in big_chungus_df['text']]
big_chungus_df['text_clean'] = [item.replace('\n',' ') for item in big_chungus_df['text_clean']]
big_chungus_df['text_clean'] = [item.replace('™',' ') for item in big_chungus_df['text_clean']]
big_chungus_df['text_clean'] = [item.replace(',',' ') for item in big_chungus_df['text_clean']]
big_chungus_df['text_clean'] = [item.replace('-',' ') for item in big_chungus_df['text_clean']]
big_chungus_df['text_clean'] = [item.replace(';',' ') for item in big_chungus_df['text_clean']]
big_chungus_df['text_clean'] = [item.replace(':',' ') for item in big_chungus_df['text_clean']]
big_chungus_df['text_clean'] = [item.replace('©',' ') for item in big_chungus_df['text_clean']]
big_chungus_df['text_clean'] = [item.replace('®',' ') for item in big_chungus_df['text_clean']]


def clean_text(text):
    clean_text = clean.clean(text, 
    fix_unicode=True, 
    to_ascii=True, 
    lower=True, 
    no_line_breaks=True,
    no_urls=True, 
    no_numbers=False, 
    no_digits=False, 
    no_currency_symbols=True, 
    no_punct=False, 
    replace_with_punct=" ", 
    replace_with_url="<URL>", 
    replace_with_number="<NUMBER>", 
    replace_with_digit="", 
    replace_with_currency_symbol="<CUR>",
    lang='en')
    return clean_text

big_chungus_df['text_clean'] = [clean_text(item) for item in big_chungus_df['text_clean']]
big_chungus_df['text_clean_no_periods'] = [item.replace('.','') for item in big_chungus_df['text_clean']]

#big_chungus_df.to_csv('grade5_corpus_clean.csv',index=False)

2628
2568


# Keywords
https://pypi.org/project/rake-nltk/

In [6]:
r = Rake() # Uses stopwords for english from NLTK, and all puntuation characters.

#big_chungus_df['text_keywords'] = [r.extract_keywords_from_text(item)]

example_list = []
for item in big_chungus_df['text_clean']:
    r.extract_keywords_from_text(item)
    ranked_keywords = r.get_ranked_phrases()
    example_list.append(ranked_keywords) 

#example_list
#r.extract_keywords_from_text(<text to process>)

#r.get_ranked_phrases() # To get keyword phrases ranked highest to lowest.

In [7]:
big_chungus_df['text_keywords'] = [item for item in example_list]

big_chungus_df['text_keywords'] = big_chungus_df.text_keywords.apply(lambda x: ' '.join([str(i) for i in x]))
#big_chungus_df['text_keywords'] = big_chungus_df.text_keywords.apply(lambda x: ', '.join([i for i in x]))
#big_chungus_df['text_keywords']

# Summarization

In [8]:
# summarizer = pipeline("summarization")

## To use the t5-base model for summarization:
summarizer_2 = pipeline("summarization", model="t5-base", tokenizer="t5-base", framework="tf")

os.environ["CUDA_VISIBLE_DEVICES"] = ""


All model checkpoint layers were used when initializing TFT5ForConditionalGeneration.

All the layers of TFT5ForConditionalGeneration were initialized from the model checkpoint at t5-base.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFT5ForConditionalGeneration for predictions without further training.


In [9]:
dir(summarizer_2)
from tensorflow.python.client import device_lib
print (device_lib.list_local_devices())

[name: "/device:CPU:0"
device_type: "CPU"
memory_limit: 268435456
locality {
}
incarnation: 16333549137087775002
]


In [10]:
#big_chungus_df['text_clean_summarized'] = [summarizer_2(big_chungus_df['text_clean'][item],max_length=100,min_length=5,do_sample=False) for item in big_chungus_df['text_clean'].index]
big_chungus_df['text_clean_summarized'] = [PlaintextParser.from_string(string,Tokenizer("english")) for string in big_chungus_df['text_clean']]
summarizer = LexRankSummarizer()
big_chungus_df['text_clean_summarized'] = [summarizer(row.document, 3) for row in big_chungus_df['text_clean_summarized']]
big_chungus_df['text_clean_summarized'] = big_chungus_df['text_clean_summarized'].astype(str)
big_chungus_df['text_clean_summarized'] = [item.replace('(<Sentence: ','') for item in big_chungus_df['text_clean_summarized']]
big_chungus_df['text_clean_summarized'] = [item.replace('<Sentence: ','') for item in big_chungus_df['text_clean_summarized']]
big_chungus_df['text_clean_summarized'] = [item.replace('>)','') for item in big_chungus_df['text_clean_summarized']]
big_chungus_df['text_clean_summarized'] = [item.replace('>, ','') for item in big_chungus_df['text_clean_summarized']]


In [11]:
#big_chungus_df['text_clean_summarized'] = [summarizer_2(big_chungus_df['text_clean'][item],max_length=100,min_length=5,do_sample=False) for item in big_chungus_df['text_clean'].index]

#clean up for t5 model
#big_chungus_df['text_clean_summarized'] = [str(item).replace("[{'summary_text': '",'') for item in big_chungus_df['text_clean_summarized']]
#big_chungus_df['text_clean_summarized'] = [item.replace("'",'') for item in big_chungus_df['text_clean_summarized']]
#big_chungus_df['text_clean_summarized'] = [item.replace("}]",'') for item in big_chungus_df['text_clean_summarized']]

# Distil-Roberta

### Embedding

In [12]:
model = SentenceTransformer('paraphrase-distilroberta-base-v1')

big_chungus_df['text_clean_embedded'] = [model.encode(item) for item in big_chungus_df['text_clean']]
big_chungus_df['text_clean_no_periods_embedded'] = [model.encode(item) for item in big_chungus_df['text_clean_no_periods']]
big_chungus_df['text_keywords_clean_embedded'] = [model.encode(item) for item in big_chungus_df['text_keywords']]
big_chungus_df['text_summarized_clean_embedded'] = [model.encode(item) for item in big_chungus_df['text_clean_summarized']]

distil_roberta_columns = ['file', 'page', 'text', 'text_clean', 'text_clean_no_periods','text_clean_embedded', 'text_clean_no_periods_embedded','text_keywords','text_keywords_clean_embedded','text_clean_summarized','text_summarized_clean_embedded']
distil_roberta_df = big_chungus_df[distil_roberta_columns]

In [13]:

#distil_roberta_df.to_csv('grade5_corpus_clean_keywords_summarized_distilroberta.csv',index=False)
distil_roberta_df.to_pickle('grade5_corpus_clean_keywords_sumy_distilroberta.pkl')

In [14]:
end = time.time()
print(end - start)

1941.1005623340607


In [15]:
print(distil_roberta_df['text_clean_summarized'][100])
print()
print(distil_roberta_df['text_clean'][100])

point out that the root phon means fisound.fl have students use the word phonograph in a sentence.materials online display and engage generative vocabulary 1.4 know it show it p. 8 instructional vocabulary root a basic word part usually from greek or latin that carries meaning prefix a word part added to the beginning of a base word that changes the meaning of the word generative vocabulary step 1 introduce the skill project display and engage generative vocabulary 1.4 .then model how to use roots and prefixes to determine the meaning of a word.

vocabulary english learner support build vocabulary ask what does a phonograph do? supply the sentence frame a plays recorded music or . point out that the root phon means fisound.fl have students use the word phonograph in a sentence. ask students to explain how the meanings of the words phonograph telephone and symphony are similar. allow them to reference a dictionary if needed. learning objectives determine the meaning of grade level acade

In [16]:
print(distil_roberta_df['text_clean_summarized'][888])
print()
print(distil_roberta_df['text_clean'][888])

c character development revise for w30 characterization w30 revise for w192 characters w21 w134 w170 w192 imagine w26 charts flow chart w186 four column chart w251 t chart w40 w191 three column chart w89 two column chart w159 w221 chronological order w5 claim w68 w69 clarity revise for w46 climax w25 w91 w171 clocking activity w16 w32 w48 w64 w82 w98 w113 w130 w146 w162 w178 w194 closing w56 w155 coherence external w161 internal w161 revise for w46 w60 coherent w60 collective nouns w225 w226 colon w338 w340w342 commas w333w337 appositives w333 connect to writing w337 other uses w334 review w336 in sentences w335 comma splice w199 w201 w202 commas and semicolons w323w327 commas with direct address and tag questions w325 connect to writing w327 introductory elements w324 punctuation in compound and complex sentences w323 review w326 commas in sentences w328w332 connect to writing w332 introductory words w328 w330 with names w329 review w331 using w330 common noun w218 w221 w222 comparati

In [17]:
pages = 40
pages = range(100)
for pages in distil_roberta_df['text_clean'][pages]:
    print(pages)

grade 5 teacher s guide volume 1
authors and advisors alma flor ada kylene beers f. isabel campoy joyce armstrong carroll nathan clemens anne cunningham martha c. hougen elena izquierdo carol jago erik palmer robert e. probst shane templeton julie washington contributing consultants david dockterman mindset works jill eggleton grade 5 volume 1 teacher s guide do not edit changes must be made through fifile infofl correctionkey=nl b
copyright 2020 by houghton mifflin harcourt publishing company all rights reserved. no part of this work may be reproduced or transmitted in any form or by any means electronic or mechanical including photocopying or recording or by any information storage or retrieval system without the prior written permission of the copyright owner unless such copying is expressly permitted by federal copyright law. permission is hereby granted to individuals using the corresponding student s textbook or kit as the major vehicle for regular classroom instruction to photoc

In [18]:
pages = 40
pages = range(pages,100)
for pages in distil_roberta_df['text_clean'][pages]:
    print(pages)

welcome to the module 1 synthesizing knowledge 1 at the beginning of the module introduce the module topic. point students to my book book 1 p. 14 and use display and engage knowledge map 1.1 to give students the first step in building their knowledge maps throughout the module. 2 after reading each text have students add to the knowledge map in their my book. at the end of each week use display and engage knowledge map 1.5 1.10 or 1.14 and discuss the added information. 3 at the end of the module students will synthesize what they have learned about the topic and make connections to self society and other texts. building knowledge networks as students read view and interact with the texts and media in this module they build deep topic knowledge about innovation perseverance and the desire to solve problems and how this information connects to their lives. display and engage knowledge map 1.14 cars sound recordings own a patent get rich be first to fly computers provide electricity mak

In [19]:
pages = 0
pages = range(0,100)
for pages in distil_roberta_df['text_clean'][pages]:
    print(pages)

grade 5 teacher s guide volume 1
authors and advisors alma flor ada kylene beers f. isabel campoy joyce armstrong carroll nathan clemens anne cunningham martha c. hougen elena izquierdo carol jago erik palmer robert e. probst shane templeton julie washington contributing consultants david dockterman mindset works jill eggleton grade 5 volume 1 teacher s guide do not edit changes must be made through fifile infofl correctionkey=nl b
copyright 2020 by houghton mifflin harcourt publishing company all rights reserved. no part of this work may be reproduced or transmitted in any form or by any means electronic or mechanical including photocopying or recording or by any information storage or retrieval system without the prior written permission of the copyright owner unless such copying is expressly permitted by federal copyright law. permission is hereby granted to individuals using the corresponding student s textbook or kit as the major vehicle for regular classroom instruction to photoc